# Build Q1 2025's channel figure

*Data Analysis in Python · Topic 8.4*

The head of e-commerce wants one image for the quarterly business review: how well each channel's sessions converted in January to March 2025, and how traffic moved week by week. Build both charts in one figure, give it a headline, and save it as a PNG that's ready to paste into a slide.

**Before you start**, choose **File → Save a copy in Drive**, and work in the copy that opens. This workbook is shared with every student, so Colab doesn't save your changes to it.

1. Run the parameters cell, the cell that builds the `sessions` table (lesson 3.'s steps), the cell that adds its `channel_group` column (topic 4.2.'s steps) and the cell that builds the `daily` table: one row per day, with the dates as the index (topic 6.1.'s steps).
2. In the first cell marked ✏️, create:
   - `conversion_by_channel`: each channel group's conversion rate, its transactions over its sessions, as a Series of fractions (0.05 for 5%)
   - `weekly_sessions`: the `daily` table's sessions totaled by Monday-to-Sunday week, with only the weeks that have all 7 days, as a Series
3. In the second cell marked ✏️, create a Figure with two charts side by side, and name them:
   - `report_fig`: the Figure, with a headline title that states something the charts show
   - `rate_ax`: the left Axes, a horizontal bar chart of `conversion_by_channel` with the highest rate on top, its value axis starting at 0 and its ticks formatted as percentages
   - `weekly_ax`: the right Axes, a line chart of `weekly_sessions`, with a title
4. In the last cell marked ✏️, save `report_fig` to `REPORT_FILE`.
5. Choose **Runtime → Restart session and run all**, and read the **Check your work** cell's output. Then open the PNG from the Files panel, and look at it the way the head of e-commerce will.

**Expected output**: `conversion_by_channel` has nine channel groups, from Paid Video's 0.64% to Email's 6.22%. `weekly_sessions` has the 12 full weeks, from the week ending January 12 to the one ending March 30, 2025, 304,708 sessions in all. `report_fig` has two Axes and a headline: `rate_ax` with nine horizontal bars, sorted with Email on top, its value axis starting at 0, and `weekly_ax` with one line of the 12 weeks. `REPORT_FILE`, `q1_2025_channels.png`, is saved from `report_fig` and isn't blank.

## Parameters

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

DATA_URL = "https://storage.googleapis.com/simmer-python-course-data/simmer-store/v1/"
MONTHS = ["2025-01", "2025-02", "2025-03"]
START, END = "2025-01-01", "2025-03-31"
REPORT_FILE = "q1_2025_channels.png"

In [ ]:
# The clean sessions table, with lesson 3's steps. Run it, and leave it as it is.
DAY_FIRST_MONTHS = ["2025-03"]
SOURCE_ALIASES = {
    "facebook.com": "facebook", "m.facebook.com": "facebook", "l.facebook.com": "facebook",
    "lm.facebook.com": "facebook", "fb": "facebook",
    "instagram.com": "instagram", "l.instagram.com": "instagram", "ig": "instagram",
}


def read_export(month):
    """Read one month's export with dates as dates, revenue as numbers and lost campaigns as missing."""
    export = pd.read_csv(DATA_URL + f"sessions_export_{month.replace('-', '_')}.csv",
                         na_values={"session_campaign": ["(not set)", "-"]})
    date_format = "%d/%m/%Y" if month in DAY_FIRST_MONTHS else "%Y-%m-%d"
    export["date"] = pd.to_datetime(export["date"], format=date_format)
    if not pd.api.types.is_numeric_dtype(export["revenue"]):
        export["revenue"] = export["revenue"].str.replace("$", "").str.replace(",", "").astype(float)
    return export


raw = pd.concat([read_export(month) for month in MONTHS], ignore_index=True)
raw["session_campaign"] = raw["session_campaign"].fillna("(not set)")
deduped = raw.drop_duplicates()
sessions = deduped[deduped["date"].between(START, END) & (deduped["session_medium"] != "test")].reset_index(drop=True)
sessions["source"] = sessions["session_source"].str.lower().str.strip().replace(SOURCE_ALIASES)
sessions.shape

In [ ]:
# Channel groups, with topic 4.2's steps and Foundations topic 12.3's function. Run it, and leave it as it is.
SOCIAL_SITES = {"facebook.com", "m.facebook.com", "l.facebook.com", "lm.facebook.com", "instagram.com", "l.instagram.com"}


def clean(value):
    """Return the value lowercased and stripped, or "" for None."""
    if value is None:
        return ""
    return value.strip().lower()


def channel_group(source, medium):
    """Return the channel group of a session's source and medium, with Foundations topic 12.3's rules."""
    source = clean(source)
    medium = clean(medium)
    if source == "(direct)" and medium == "(none)":
        return "Direct"
    if medium == "email":
        return "Email"
    if medium == "cpc":
        return "Paid Search"
    if medium == "display":
        return "Display"
    if medium == "paid_social":
        return "Paid Social"
    if medium == "video":
        return "Paid Video"
    if medium == "organic":
        return "Organic Search"
    if medium == "referral" and source in SOCIAL_SITES:
        return "Organic Social"
    if medium == "referral":
        return "Referral"
    return "Unassigned"


def row_channel_group(row):
    """Return the channel group of one row of the sessions table."""
    return channel_group(row["session_source"], row["session_medium"])


sessions["channel_group"] = sessions.apply(row_channel_group, axis=1)
sessions["channel_group"].value_counts()

In [ ]:
# One row per day, with the dates as the index (topic 6.1). Run it, and leave it as it is.
daily = sessions.groupby("date")[["sessions", "transactions", "revenue"]].sum()
daily.shape

## The two tables

In [ ]:
# ✏️ Create conversion_by_channel and weekly_sessions


## The figure

In [ ]:
# ✏️ Create report_fig, rate_ax and weekly_ax


## The file

In [ ]:
# ✏️ Save report_fig to REPORT_FILE


In [ ]:
# @title Check your work
# @markdown This cell runs last, with the others. Run it with Runtime → Restart session and run all.
# This cell only reads your variables, your charts, your file and the cells that have run. It doesn't change anything.
def check_your_work():
    import os
    import sys
    import matplotlib.axes
    import matplotlib.figure
    import matplotlib.pyplot as plt
    import pandas as pd
    found = globals()
    results = []

    inputs = found.get("In")
    history = [source for source in inputs[1:-1] if "def check_your_work" not in source] if isinstance(inputs, list) else None
    checked_before = isinstance(inputs, list) and any("def check_your_work" in source for source in inputs[1:-1])
    if history is None:
        results.append((None, "This check can't see which cells have run, so it can't tell whether this was a clean run. Choose Runtime → Restart session and run all to be sure."))
    else:
        repeated = [source for source in history if source.strip() and history.count(source) > 1]
        error = getattr(sys, "last_exc", None) or getattr(sys, "last_value", None)
        if checked_before:
            results.append((None, "This cell has run before in this session, so the cells above may have changed since they ran. Choose Runtime → Restart session and run all to check the notebook as it is now."))
        elif repeated:
            first_line = repeated[0].strip().splitlines()[0]
            results.append((None, f"Some cells ran more than once in this session, such as the one that starts with {first_line!r}. Choose Runtime → Restart session and run all, so every cell runs once, from the top."))
        elif error is not None:
            results.append((None, f"A cell stopped with {type(error).__name__} earlier in this session, so the cells after it may not have run in order. Choose Runtime → Restart session and run all."))
        else:
            results.append((True, "Every cell in this session ran once, with no errors."))

    def bars(ax):
        """Return the bars' values from bottom to top, or left to right, and whether they're horizontal."""
        patches = [patch for patch in ax.patches if patch.get_width() and patch.get_height()]
        if not patches:
            return [], None
        horizontal = len({round(patch.get_height(), 6) for patch in patches}) == 1 and len({round(patch.get_width(), 6) for patch in patches}) > 1
        if horizontal:
            ordered = sorted(patches, key=lambda patch: patch.get_y())
            ordered = ordered[::-1] if ax.yaxis_inverted() else ordered
            return [patch.get_width() for patch in ordered], True
        ordered = sorted(patches, key=lambda patch: patch.get_x())
        ordered = ordered[::-1] if ax.xaxis_inverted() else ordered
        return [patch.get_height() for patch in ordered], False

    rates = found.get("conversion_by_channel")
    weekly = found.get("weekly_sessions")
    if isinstance(rates, pd.DataFrame) and rates.shape[1] == 1:
        rates = rates.iloc[:, 0]
    if isinstance(weekly, pd.DataFrame) and weekly.shape[1] == 1:
        weekly = weekly.iloc[:, 0]
    if not isinstance(rates, pd.Series) or "Email" not in rates.index:
        results.append((False, "conversion_by_channel should be a Series with one rate per channel group: total transactions and sessions by channel_group, then divide."))
    elif abs(rates["Email"] - 0.0577) < 0.0001:
        results.append((False, "conversion_by_channel's Email rate is 5.77%: that's the average of each row's rate, which counts a campaign with 3 sessions as much as one with 3,000. Total transactions and sessions by channel group first, then divide (topic 4.4)."))
    elif abs(rates["Email"] - 6.22) < 0.01:
        results.append((False, "conversion_by_channel is in percent: Email is 6.22. Keep rates as fractions, 0.0622, and format them as percentages on the chart."))
    elif len(rates) != 9 or abs(rates["Email"] - 0.0622) > 0.0001 or abs(rates["Paid Video"] - 0.0064) > 0.0001:
        results.append((False, f"conversion_by_channel should have nine channel groups, with Email at about 0.0622 and Paid Video at about 0.0064. It has {len(rates)}, with Email at {rates['Email']:.4f}."))
    elif not isinstance(weekly, pd.Series):
        results.append((False, "weekly_sessions should be a Series: daily[\"sessions\"].resample(\"W\").sum(), with the partial weeks left out."))
    elif len(weekly) == 14:
        results.append((False, "weekly_sessions has 14 weeks: the first and last are partial, January 1–5 and March 31. Keep only the weeks with 7 days of data (topic 6.2)."))
    elif len(weekly) != 12 or weekly.sum() != 304708:
        results.append((False, f"weekly_sessions should have the 12 full weeks, adding up to 304,708 sessions. It has {len(weekly)}, adding up to {weekly.sum():,}."))
    else:
        results.append((True, f"Nine conversion rates, from Paid Video's {rates.min():.2%} to Email's {rates.max():.2%}, and 12 full weeks of sessions, {weekly.sum():,} in all."))

    fig = found.get("report_fig")
    rate_ax, weekly_ax = found.get("rate_ax"), found.get("weekly_ax")
    if not isinstance(fig, matplotlib.figure.Figure):
        results.append((False, f"report_fig should be the Figure: report_fig, (rate_ax, weekly_ax) = plt.subplots(1, 2, figsize=(13, 4.5)). It is a {type(fig).__name__}."))
    elif len(fig.axes) != 2 or rate_ax not in fig.axes or weekly_ax not in fig.axes:
        results.append((False, f"report_fig should have two Axes, rate_ax on the left and weekly_ax on the right, from one plt.subplots(1, 2). It has {len(fig.axes)}, and rate_ax and weekly_ax should be them."))
    else:
        values, horizontal = bars(rate_ax)
        lines = weekly_ax.get_lines()
        if len(values) != 9:
            results.append((False, f"rate_ax should have nine bars, one per channel group. It has {len(values)}."))
        elif horizontal is False:
            results.append((False, "rate_ax's bars are vertical. Make them horizontal, so the channel names fit: plot.barh(ax=rate_ax)."))
        elif abs(max(values) - 0.0622) > 0.0001:
            results.append((False, f"rate_ax's bars should be conversion_by_channel's rates, with Email's 0.0622 the longest. The longest is {max(values):,.4f}."))
        elif values != sorted(values):
            results.append((False, "rate_ax's bars should be sorted with the highest rate on top: conversion_by_channel.sort_values().plot.barh(ax=rate_ax)."))
        elif rate_ax.get_xlim()[0] != 0:
            results.append((False, f"rate_ax's value axis starts at {rate_ax.get_xlim()[0]:.4f}, so the bars' lengths don't compare the rates. Start it at 0: rate_ax.set_xlim(left=0)."))
        elif "%" not in "".join(label.get_text() for label in rate_ax.get_xticklabels()):
            results.append((False, "rate_ax's ticks aren't percentages. Format them: rate_ax.xaxis.set_major_formatter(\"{x:.0%}\")."))
        elif len(lines) != 1 or len(lines[0].get_ydata()) != 12 or abs(sum(lines[0].get_ydata()) - 304708) > 0.5:
            results.append((False, "weekly_ax should have one line, weekly_sessions' 12 weeks: weekly_sessions.plot(ax=weekly_ax)."))
        elif not weekly_ax.get_title().strip():
            results.append((False, "weekly_ax has no title. Say what it shows: weekly_ax.set_title(\"Sessions by week\")."))
        elif fig.get_suptitle().strip() == "":
            results.append((False, "report_fig has no headline. Add one that states what the charts show: report_fig.suptitle(\"…\")."))
        else:
            results.append((True, f"report_fig has the rates, sorted from 0 with percentage ticks, and the 12 weeks, under the headline {fig.get_suptitle()!r}."))

    report_file = found.get("REPORT_FILE", "q1_2025_channels.png")
    saved = isinstance(history, list) and any("savefig" in source for source in history)
    if not os.path.exists(report_file) or not saved:
        results.append((False, f"{report_file} hasn't been saved in this session. Save the Figure: report_fig.savefig(REPORT_FILE, dpi=150, bbox_inches=\"tight\")."))
    else:
        image = plt.imread(report_file)
        if image[..., :3].std() == 0:
            results.append((False, f"{report_file} is blank: one color, nothing else. plt.savefig() in a later cell saves a new, empty Figure, because Colab closes a Figure once it's shown. Save it by name: report_fig.savefig(REPORT_FILE)."))
        elif image.shape[1] < 1000:
            results.append((False, f"{report_file} is only {image.shape[1]} pixels wide. Save it from report_fig, which is two charts wide, with dpi=150."))
        else:
            results.append((True, f"{report_file} is saved, {image.shape[1]} × {image.shape[0]} pixels, and it isn't blank. Open it from the Files panel before you send it."))

    for passed, message in results:
        print({True: "✅ ", False: "❌ ", None: "⚠️ "}[passed] + message)
    if any(passed is False for passed, message in results):
        print("\nFix the first ❌, then choose Runtime → Restart session and run all.")
    elif any(passed is None for passed, message in results):
        print("\nYour results match. Restart the session and run all, to check them in a clean run.")
    else:
        print("\nEverything matches, in a clean run.")


check_your_work()

<details>
<summary><b>Show solution</b></summary>

Try the exercise yourself first. This is one way to solve it: if the check passes, your way is right too.

```python
by_channel = sessions.groupby("channel_group")[["sessions", "transactions"]].sum()
conversion_by_channel = by_channel["transactions"] / by_channel["sessions"]
days = daily["sessions"].resample("W").count()
weekly_sessions = daily["sessions"].resample("W").sum()[days == 7]
conversion_by_channel.sort_values(), len(weekly_sessions)
```

```python
report_fig, (rate_ax, weekly_ax) = plt.subplots(1, 2, figsize=(13, 4.5))
conversion_by_channel.sort_values().plot.barh(ax=rate_ax)
rate_ax.bar_label(rate_ax.containers[0], fmt="{:.1%}", padding=3)
rate_ax.xaxis.set_major_formatter("{x:.0%}")
rate_ax.set_title("Conversion rate by channel group")
rate_ax.set_ylabel("")
weekly_sessions.plot(ax=weekly_ax, marker="o")
weekly_ax.yaxis.set_major_formatter("{x:,.0f}")
weekly_ax.set_ylim(bottom=0)
weekly_ax.set_title("Sessions by week (weeks ending Sunday)")
weekly_ax.set_xlabel("")
report_fig.suptitle("Q1 2025: Email converted at 6.2%, ten times Paid Video's rate", fontsize=14, x=0.01, ha="left")
report_fig.tight_layout()
plt.show()
```

```python
report_fig.savefig(REPORT_FILE, dpi=150, bbox_inches="tight")
```

</details>